## 1. Executive Overview

StatGenie is an AI-powered data intelligence system that accelerates Exploratory Data Analysis (EDA), cleaning, feature preparation, visualization, and narrative insight generation. It blends deterministic data tooling (Pandas, Plotly) with LLM augmentation (Gemini) to produce explainable analytics artifacts (JSON outputs, dynamic slicers, charts, PDF reports).

The current repository implements a streamlined subset (≈ v3.0 baseline):
- Multi-format ingestion (CSV, Excel, PDF, DOCX, TXT, Image via OCR + LLM structuring)
- Data cleaning (missing fill, date normalization, IQR-based outlier capping)
- Automated analysis (numeric/categorical summaries, KPIs, randomized chart set, AI data story)
- Dynamic slicers and filter re-analysis endpoint (/filters)
- PDF report generation (charts + KPIs + narrative)
- JSON-safe serialization (custom encoder)
Missing layers for full v3.6: advanced feature engineering, statistical test suite, RAG layer, Redis-backed job persistence, multi-source import (Sheets/SQL/API), chart explain mode, AI Copilot conversational interface, export variants (Excel/HTML/PNG ZIP), filter preview, robust config & orchestration modes (automated vs semi-automated).

## 2. Version Evolution Summary

| Version | Focus | Highlights | Status in Repo |
|---------|-------|-----------|----------------|
| 1.0 | Foundational EDA | Basic stats, matplotlib charts, PDF export | NOT PRESENT (superseded) |
| 2.0 | Data Prep + Semi-Auto | Cleaning model, feature engineering, stats tests, config-driven pipeline | PARTIALLY MISSING (feature engineering & stats absent) |
| 3.0 | Full Automation | End-to-end pipeline, Plotly charts, AI narrative, Redis jobs | PARTIAL (charts + narrative exist; Redis absent) |
| 3.5 | Interactivity + RAG | Smart filters, RAG chatbot, dynamic chart explorer | MISSING (simple filters only) |
| 3.6 | AI-Oriented Orchestration | Copilot, multi-source imports, export suite, chart explain mode | TARGET STAGE |

The current codebase aligns closest with early 3.0 capabilities but lacks persistence, extensible orchestration, and AI interaction layers beyond a single narrative generation call.

## 3. Current Implementation Inventory

### 3.1 Core Modules
- `app.py`: Flask API endpoints (`/clean_and_analyze`, `/filters`, `/download_report`, health, upload page)
- `data_cleaning_model.py`: Missing value imputation, date normalization, Winsorization report
- `data_analysis.py`: Summaries, KPIs, randomized chart generation, slicers, data story (Gemini), filter application logic
- `file_processors/`: Automated ingestion across formats with AI fallback extraction (`automated_load`)
- `image_handler.py` / `unstructured_handler.py`: LLM-powered OCR / text-to-CSV conversion
- `pdf_report.py`: ReportLab + Plotly → multi-section PDF with charts
- `json_encoder.py`: Recursive sanitization of numpy/pandas/plotly types
- `Dockerfile`: Multi-stage build (builder + runtime), system libs for PDF/OCR/charts
### 3.2 Implemented API Surface
| Endpoint | Method | Purpose | Notes |
|----------|--------|---------|-------|
| `/` | GET | Health JSON | Basic message |
| `/upload_page` | GET | Simple HTML uploader | Dev/testing only |
| `/clean_and_analyze` | POST | Upload, clean, analyze, return JSON | In-memory job tracking (UUID) |
| `/filters` | POST | Apply filters to existing job | Limited filter expression model |
| `/download_report` | POST | Generate PDF report from job | Only PDF format supported |

### 3.3 In-Memory Job Store
- Simple dict: `jobs[job_id] = { 'df': cleaned_df, 'report': cleaning_report }`
- No expiration, persistence, concurrency control, or recovery
- Not scalable for multi-user scenarios
### 3.4 Chart Pipeline
- Random chart candidate selection (diverse chart types)
- Plotly JSON sanitized for frontend consumption
- KPI extraction limited to basic row/column metrics and first numeric columns
### 3.5 AI Usage
- Single-pass narrative generation using Gemini model (`gemini-2.5-pro`)
- No structured prompt versioning, caching, or fallback model strategy
- No interactive Q&A / Copilot state
### 3.6 Data Cleaning Coverage
- Missing: Typo correction, semantic type inference, mixed-format column normalization beyond 'date' substring
- Present: Median/mode fill, date coercion, IQR capping (Winsorization)
### 3.7 Missing Modules Referenced in v3.6 Vision
The following logical components are absent and must be created:
- `feature_engineer.py` (scaling, encoding, derived metrics)
- `stats_module.py` (correlation, ANOVA, chi-square, t-test)
- `rag_agent.py` + vector store integration (e.g., Qdrant / FAISS / Redis embeddings)
- `job_storage.py` (Redis abstraction, TTL, compression, fallback)
- `config.py` (centralized settings: model names, limits, feature flags)
- Multi-source import adapters: `sheets_import.py`, `sql_import.py`, `api_import.py`
- Exporters: `export_excel.py`, `export_html.py`, `export_images.py`
- Filter preview service: `filter_preview.py`
- Chart explain service: `chart_explain.py`
- Copilot conversational orchestrator: `copilot.py`
- Async layer / background tasks (Celery or RQ optional)

## 4. Gap Analysis (Current vs Target v3.6)

| Capability | Current | Target v3.6 | Required Action |
|-----------|---------|-------------|-----------------|
| Job Persistence | In-memory dict | Redis w/ TTL, compression, fallback | Implement `job_storage.py` + config |
| Multi-Source Import | File uploads only | Sheets / SQL / API ingestion | Add import adapters + endpoints |
| Feature Engineering | None | Scaling, encoding, derived metrics | Create `feature_engineer.py` |
| Statistical Tests | None | Correlation, ANOVA, chi-square, t-test | Build `stats_module.py` |
| Advanced Filters | Basic equality/inclusion | Rich operators + preview | Filter DSL + preview endpoint |
| Filter Preview | Absent | Size prediction before apply | Implement sampler function |
| Export Formats | PDF only | PDF / HTML / XLSX / PNG ZIP | Add modular exporters |
| Chart Explainability | None | AI summary per chart | `chart_explain.py` reading chart spec |
| AI Copilot | One-off narrative | Stateful conversational assistant | `copilot.py` + memory store |
| RAG Layer | None | Vector search augmenting prompts | `rag_agent.py` + embedding store |
| Config Management | Hard-coded env checks | Centralized hierarchical config | Introduce `config.py` |
| Observability | Basic logging | Structured logs + metrics | Logging config + Prometheus hooks |
| Async Processing | Synchronous | Optional async / queued | Evaluate Celery/RQ or Flask-SocketIO |
| Security | None (open endpoints) | Auth + rate limits | Add auth middleware & quotas |
| Prompt Governance | Ad-hoc strings | Versioned templates + safety filters | Prompt registry module |
| Chart Determinism | Random subset | Deterministic + user selection + categories | Chart catalog + selection API |
| Memory Hygiene | No expiry | Job TTL + cleanup scheduler | Redis TTL + periodic sweep |

## 5. Target Architecture (v3.6 Logical Layering)

````text
Presentation Layer:
  - React (planned) or current HTML: Upload UI, Filters, Chart Explorer, Copilot Chat
  - REST + (future) WebSocket for live events
Service Layer (Flask):
  - Controllers: upload, analyze, filters, exports, copilot, rag_chat
  - Orchestrators: automated_pipeline, semi_pipeline
Domain Layer:
  - CleaningModel, FeatureEngineer, StatsModule, ChartBuilder, FilterEngine
  - Copilot (stateful AI), RAGAgent (embedding retrieval)
,
,
,
,
]},{
:
,
:{
:
},
:[
6
,
,
6.1
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
6.2
,
,
,
,
,
,
,
,
,
,
,
,
,
,
]},{
:
,
:{
:
},
:[
7
,
,
1
1
,
,
2
3
,
,
3
5
,
,
4
7
,
,
5
9
,
,
6
11
,
,
]},{
:
,
:{
:
},
:[
9
,
,
,
,
,
,
,
,
,
]},{
:
,
:{
:
},
:[
10
,
,
,
,
,
,
,
,
,
,
]},{
:
,
:{
:
},
:[
11
,
,
1
,
 ]},{
:
,
:{
:
},
:[
12
,
,
,
,
,
,